# EE/CS 148B HW3: Colab Setup + Full Pipeline

This notebook (a) sets up the Colab environment, (b) writes implementations for every HW3 stub into the repo on Drive, and (c) drives §2 ViT through §6 RoPE end-to-end.

## Colab Setup

Before running:

- Switch the runtime to a GPU runtime (A100 or H100 recommended).
- This notebook clones the repo from GitHub into `/content/hw3` — no Drive needed.
- At the end, all `runs/` outputs are zipped and auto-downloaded.

Note: We will be using `pip` for dependencies inside Colab.

In [ ]:
%%capture
!pip -q install -U torch torchvision numpy==1.26.0 transformers datasets "sentence-transformers<4.0" accelerate pillow tqdm matplotlib wandb pyyaml einops pytest pytest-cov
!pip -q install ninja packaging
!pip -q install flash-attn --no-build-isolation

In [ ]:
!pip -q install numpy==1.26.0 --force-reinstall

In [ ]:
# Clone the repo locally (no Drive). Pulls fresh if rerun.
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/itsyashk/vision-language-models.git'
REPO_ROOT = Path('/content/hw3')

if REPO_ROOT.exists():
    print('Repo already cloned, pulling latest...')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

assert REPO_ROOT.exists(), f'Clone failed: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


In [ ]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

## Write Implementations into the Repo

Each `%%writefile` cell overwrites a stub file in the repo on Drive with a working implementation. Run them top-to-bottom before importing.

In [ ]:
%%writefile basics/vit.py
import torch
import torch.nn as nn

from basics.model import Block


class PatchEmbeddings(nn.Module):
    def __init__(self, img_size: int, patch_size: int, d_model: int):
        super().__init__()
        assert img_size % patch_size == 0, 'img_size must be divisible by patch_size'
        self.img_size = img_size
        self.patch_size = patch_size
        self.d_model = d_model
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.proj = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)            # (B, d_model, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, N, d_model)
        return x


class ViT(nn.Module):
    def __init__(
        self,
        img_size: int,
        patch_size: int,
        d_model: int,
        num_heads: int,
        num_blocks: int,
        dropout: float = 0.1,
        positional_encoding: str = 'learned',  # 'learned' | 'none'
    ):
        super().__init__()
        self.patch_embed = PatchEmbeddings(img_size, patch_size, d_model)
        self.num_patches = self.patch_embed.num_patches
        self.grid_size = self.patch_embed.grid_size
        self.d_model = d_model
        self.positional_encoding = positional_encoding
        block_size = self.num_patches + 1
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        if positional_encoding == 'learned':
            self.pos_embed = nn.Parameter(torch.zeros(1, block_size, d_model))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        else:
            self.pos_embed = None
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model, num_heads, block_size=block_size, is_decoder=False, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(d_model)

    def interpolate_pos_embed(self, new_num_patches: int) -> torch.Tensor:
        '''Bilinear-interpolate the learned patch positional embedding to a new grid.
        Keeps the CLS positional embedding separate.'''
        assert self.pos_embed is not None
        old_grid = self.grid_size
        new_grid = int(new_num_patches ** 0.5)
        cls_pe = self.pos_embed[:, :1]
        patch_pe = self.pos_embed[:, 1:]
        patch_pe = patch_pe.reshape(1, old_grid, old_grid, self.d_model).permute(0, 3, 1, 2)
        patch_pe = torch.nn.functional.interpolate(patch_pe, size=(new_grid, new_grid), mode='bilinear', align_corners=False)
        patch_pe = patch_pe.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, self.d_model)
        return torch.cat([cls_pe, patch_pe], dim=1)

    def forward(self, x: torch.Tensor, return_all_tokens: bool = False) -> torch.Tensor:
        B = x.shape[0]
        x = self.patch_embed(x)                          # (B, N, d)
        N = x.shape[1]
        cls = self.cls_token.expand(B, -1, -1)            # (B, 1, d)
        x = torch.cat([cls, x], dim=1)                    # (B, N+1, d)
        if self.positional_encoding == 'learned':
            if N + 1 == self.pos_embed.shape[1]:
                x = x + self.pos_embed
            else:
                x = x + self.interpolate_pos_embed(N)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        if return_all_tokens:
            return x                                      # (B, N+1, d)
        return x[:, 0]                                    # (B, d)


In [ ]:
%%writefile basics/lora.py
import math
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.rank = rank
        self.alpha = alpha
        in_features = base_layer.in_features
        out_features = base_layer.out_features
        self.A = nn.Parameter(torch.empty(rank, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)
        lora_out = (x @ self.A.t()) @ self.B.t()
        return base_out + (self.alpha / self.rank) * lora_out


def apply_lora_to_attention(model: nn.Module, rank: int, alpha: float) -> nn.Module:
    '''Replace q_proj and v_proj inside every basics.model.Head with LoRALinear wrappers.'''
    from basics.model import Head
    for module in model.modules():
        if isinstance(module, Head):
            module.q_proj = LoRALinear(module.q_proj, rank, alpha)
            module.v_proj = LoRALinear(module.v_proj, rank, alpha)
    return model


In [ ]:
%%writefile basics/rope.py
import torch
import torch.nn as nn


def _rotate_half_pairs(x: torch.Tensor) -> torch.Tensor:
    # interpret last dim as pairs (x_{2i}, x_{2i+1}) and return (-x_{2i+1}, x_{2i})
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    out = torch.stack((-x2, x1), dim=-1)
    return out.flatten(-2)


def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # x: (..., T, d), cos/sin: (T, d//2). Expand to (T, d) by interleaving.
    cos_e = torch.repeat_interleave(cos, 2, dim=-1)
    sin_e = torch.repeat_interleave(sin, 2, dim=-1)
    return x * cos_e + _rotate_half_pairs(x) * sin_e


class RoPE1D(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, base: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)              # (max_seq_len, head_dim//2)
        self.register_buffer('cos', freqs.cos(), persistent=False)
        self.register_buffer('sin', freqs.sin(), persistent=False)

    def forward(self, x: torch.Tensor, positions: torch.Tensor) -> torch.Tensor:
        cos = self.cos[positions]
        sin = self.sin[positions]
        return _apply_rope(x, cos, sin)


class RoPE2D(nn.Module):
    def __init__(self, head_dim: int, grid_size: int, base: float = 10000.0):
        super().__init__()
        assert head_dim % 4 == 0, 'head_dim must be divisible by 4 for 2D RoPE'
        self.head_dim = head_dim
        self.half = head_dim // 2
        self.grid_size = grid_size
        self.rope_x = RoPE1D(self.half, grid_size, base)
        self.rope_y = RoPE1D(self.half, grid_size, base)

    def forward(self, x: torch.Tensor, x_coords: torch.Tensor, y_coords: torch.Tensor) -> torch.Tensor:
        x_part = x[..., : self.half]
        y_part = x[..., self.half :]
        x_part = self.rope_x(x_part, x_coords)
        y_part = self.rope_y(y_part, y_coords)
        return torch.cat([x_part, y_part], dim=-1)


In [ ]:
%%writefile vlm/clip.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class ProjectionHeads(nn.Module):
    def __init__(self, d_image: int, d_text: int, d_proj: int = 256):
        super().__init__()
        self.image_proj = nn.Linear(d_image, d_proj, bias=False)
        self.text_proj = nn.Linear(d_text, d_proj, bias=False)

    def forward(self, image_embeds: torch.Tensor, text_embeds: torch.Tensor):
        img = self.image_proj(image_embeds)
        txt = self.text_proj(text_embeds)
        img = F.normalize(img, dim=-1)
        txt = F.normalize(txt, dim=-1)
        return img, txt


def init_logit_scale() -> nn.Parameter:
    return nn.Parameter(torch.tensor(math.log(1.0 / 0.07)))


def clip_loss(image_embeds: torch.Tensor, text_embeds: torch.Tensor, logit_scale: torch.Tensor) -> torch.Tensor:
    scale = logit_scale.exp()
    logits = scale * (image_embeds @ text_embeds.t())
    targets = torch.arange(logits.size(0), device=logits.device)
    loss_i = F.cross_entropy(logits, targets)
    loss_t = F.cross_entropy(logits.t(), targets)
    return 0.5 * (loss_i + loss_t)


In [ ]:
%%writefile vlm/projector.py
import torch
import torch.nn as nn


class VisionLanguageProjector(nn.Module):
    def __init__(self, d_image: int, d_decoder: int, expansion: int = 4):
        super().__init__()
        hidden = expansion * d_image
        self.fc1 = nn.Linear(d_image, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, d_decoder)

    def forward(self, image_features: torch.Tensor) -> torch.Tensor:
        if image_features.dim() == 2:
            image_features = image_features.unsqueeze(1)  # (B, 1, d_image)
        h = self.fc1(image_features)
        h = self.act(h)
        return self.fc2(h)


In [ ]:
%%writefile vlm/model.py
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

from vlm.masking import build_image_bidir_mask


class VisionLanguageModel(nn.Module):
    def __init__(self, vit, projector, decoder, tokenizer, image_token_id: Optional[int] = None):
        super().__init__()
        self.vit = vit
        self.projector = projector
        self.decoder = decoder
        self.tokenizer = tokenizer
        self.image_token_id = image_token_id

    def _encode_image(self, images: torch.Tensor, injection: str) -> torch.Tensor:
        if injection == 'cls':
            feats = self.vit(images, return_all_tokens=False)  # (B, d)
            return self.projector(feats)                        # (B, 1, D)
        else:
            feats = self.vit(images, return_all_tokens=True)   # (B, N+1, d)
            return self.projector(feats)                        # (B, N+1, D)

    def _embed_text(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.decoder.get_input_embeddings()(input_ids)

    def _fuse(self, visual_embeds, text_embeds, input_ids, attention_mask, injection):
        B = text_embeds.shape[0]
        if injection in ('cls', 'all_patches'):
            inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
            n_vis = visual_embeds.shape[1]
            vis_mask = torch.ones(B, n_vis, dtype=attention_mask.dtype, device=attention_mask.device)
            new_attn = torch.cat([vis_mask, attention_mask], dim=1)
            return inputs_embeds, new_attn, n_vis
        elif injection == 'interleaved':
            assert self.image_token_id is not None, 'image_token_id required for interleaved injection'
            n_vis = visual_embeds.shape[1]
            embeds_list = []
            attn_list = []
            for b in range(B):
                ids = input_ids[b]
                am = attention_mask[b]
                te = text_embeds[b]
                idx = (ids == self.image_token_id).nonzero(as_tuple=True)[0]
                if len(idx) == 0:
                    fused = torch.cat([visual_embeds[b], te], dim=0)
                    a = torch.cat([torch.ones(n_vis, dtype=am.dtype, device=am.device), am], dim=0)
                else:
                    i = idx[0].item()
                    fused = torch.cat([te[:i], visual_embeds[b], te[i+1:]], dim=0)
                    a = torch.cat([am[:i], torch.ones(n_vis, dtype=am.dtype, device=am.device), am[i+1:]], dim=0)
                embeds_list.append(fused)
                attn_list.append(a)
            T = max(e.shape[0] for e in embeds_list)
            D = visual_embeds.shape[-1]
            padded = torch.zeros(B, T, D, dtype=visual_embeds.dtype, device=visual_embeds.device)
            padded_attn = torch.zeros(B, T, dtype=attention_mask.dtype, device=attention_mask.device)
            for b, (e, a) in enumerate(zip(embeds_list, attn_list)):
                padded[b, :e.shape[0]] = e
                padded_attn[b, :a.shape[0]] = a
            return padded, padded_attn, n_vis
        else:
            raise ValueError(f'unknown injection {injection}')

    def _shift_labels(self, labels, n_vis, injection, input_ids):
        if labels is None:
            return None
        B, T = labels.shape
        device = labels.device
        if injection in ('cls', 'all_patches'):
            vis_labels = torch.full((B, n_vis), -100, dtype=labels.dtype, device=device)
            return torch.cat([vis_labels, labels], dim=1)
        else:
            out = []
            for b in range(B):
                idx = (input_ids[b] == self.image_token_id).nonzero(as_tuple=True)[0]
                lb = labels[b]
                if len(idx) == 0:
                    out.append(torch.cat([torch.full((n_vis,), -100, dtype=labels.dtype, device=device), lb]))
                else:
                    i = idx[0].item()
                    out.append(torch.cat([lb[:i], torch.full((n_vis,), -100, dtype=labels.dtype, device=device), lb[i+1:]]))
            T2 = max(o.shape[0] for o in out)
            padded = torch.full((B, T2), -100, dtype=labels.dtype, device=device)
            for b, o in enumerate(out):
                padded[b, :o.shape[0]] = o
            return padded

    def forward(
        self,
        images: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        injection: str = 'cls',
        mask_mode: str = 'causal',
    ):
        visual_embeds = self._encode_image(images, injection)
        visual_embeds = visual_embeds.to(self.decoder.dtype)
        text_embeds = self._embed_text(input_ids)
        inputs_embeds, new_attn, n_vis = self._fuse(visual_embeds, text_embeds, input_ids, attention_mask, injection)
        shifted_labels = self._shift_labels(labels, n_vis, injection, input_ids)

        kwargs = dict(inputs_embeds=inputs_embeds, attention_mask=new_attn)
        if mask_mode == 'image_bidir':
            B, T, _ = inputs_embeds.shape
            n_text = T - n_vis
            mask4d = build_image_bidir_mask(n_vis, n_text, inputs_embeds.device, inputs_embeds.dtype)
            mask4d = mask4d.expand(B, -1, -1, -1)
            kwargs['attention_mask'] = mask4d
        if shifted_labels is not None:
            kwargs['labels'] = shifted_labels
        return self.decoder(**kwargs)

    @torch.no_grad()
    def generate(
        self,
        images: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        injection: str = 'cls',
        mask_mode: str = 'causal',
        max_new_tokens: int = 32,
    ):
        visual_embeds = self._encode_image(images, injection)
        visual_embeds = visual_embeds.to(self.decoder.dtype)
        text_embeds = self._embed_text(input_ids)
        inputs_embeds, new_attn, n_vis = self._fuse(visual_embeds, text_embeds, input_ids, attention_mask, injection)
        out = self.decoder.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=new_attn,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
        )
        return self.tokenizer.batch_decode(out, skip_special_tokens=True)


In [ ]:
%%writefile scripts/pretrain_clip.py
import argparse
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm

from basics.text_encoder import FrozenTextEncoder
from basics.vit import ViT
from vlm.clip import ProjectionHeads, clip_loss, init_logit_scale
from vlm.data import EUROSAT_CLASSES, build_eurosat_loaders
from vlm.eval import zeroshot_classification_accuracy


def cosine_with_warmup(optimizer, warmup, total):
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', type=str, required=True)
    ap.add_argument('--output-dir', type=str, required=True)
    ap.add_argument('--positional-encoding', type=str, default='learned')
    ap.add_argument('--epochs', type=int, default=None)
    args = ap.parse_args()
    cfg = yaml.safe_load(open(args.config))
    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = args.epochs or cfg.get('num_epochs', 20)

    train_dl, val_dl, _ = build_eurosat_loaders(
        img_size=cfg['img_size'], batch_size=cfg['batch_size'])
    vit = ViT(cfg['img_size'], cfg['patch_size'], cfg['d_model'], cfg['num_heads'],
              cfg['num_blocks'], dropout=cfg.get('dropout', 0.1),
              positional_encoding=args.positional_encoding).to(device)
    text_encoder = FrozenTextEncoder('sentence-transformers/all-MiniLM-L6-v2').to(device)
    text_encoder.eval()
    proj = ProjectionHeads(cfg['d_model'], 384, cfg.get('d_proj', 256)).to(device)
    logit_scale = init_logit_scale().to(device)

    params = list(vit.parameters()) + list(proj.parameters()) + [logit_scale]
    opt = AdamW(params, lr=cfg['lr'], weight_decay=cfg.get('weight_decay', 0.1))
    steps_per_epoch = len(train_dl)
    total = steps_per_epoch * epochs
    sched = cosine_with_warmup(opt, cfg.get('warmup_steps', 200), total)

    class_prompts = [f'a satellite image of {c}' for c in EUROSAT_CLASSES]
    metrics = {'train_loss': [], 'val_acc': []}
    best = 0.0
    for epoch in range(epochs):
        vit.train(); proj.train()
        for imgs, captions in tqdm(train_dl, desc=f'epoch {epoch}'):
            imgs = imgs.to(device)
            with torch.no_grad():
                text_embeds_raw = text_encoder(list(captions))
            img_embeds_raw = vit(imgs)
            img_embeds, txt_embeds = proj(img_embeds_raw, text_embeds_raw)
            with torch.no_grad():
                logit_scale.clamp_(max=math.log(100.0))
            loss = clip_loss(img_embeds, txt_embeds, logit_scale)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            metrics['train_loss'].append(float(loss.item()))
        vit.eval(); proj.eval()
        acc = zeroshot_classification_accuracy(vit, proj, text_encoder, val_dl, class_prompts, list(range(len(EUROSAT_CLASSES))), device)
        metrics['val_acc'].append(acc)
        print(f'epoch {epoch}: val_acc={acc:.4f}')
        if acc > best:
            best = acc
            torch.save({'vit': vit.state_dict(), 'proj': proj.state_dict(),
                        'logit_scale': logit_scale.detach().cpu(),
                        'cfg': cfg, 'positional_encoding': args.positional_encoding},
                       out / 'best.pt')
    json.dump(metrics, open(out / 'metrics.json', 'w'))
    plt.figure(); plt.plot(metrics['train_loss']); plt.xlabel('step'); plt.ylabel('train loss'); plt.savefig(out / 'train_loss.png'); plt.close()
    plt.figure(); plt.plot(metrics['val_acc']); plt.xlabel('epoch'); plt.ylabel('zero-shot val acc'); plt.savefig(out / 'val_acc.png'); plt.close()
    print('Best val acc:', best)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile scripts/finetune_resisc.py
import argparse
import json
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
import yaml
from torch.optim import AdamW
from tqdm import tqdm

from basics.lora import apply_lora_to_attention
from basics.vit import ViT
from vlm.data import build_resisc45_loaders


def build_model(cfg, pretrained, method, rank, alpha, device):
    vit = ViT(cfg['img_size'], cfg['patch_size'], cfg['d_model'], cfg['num_heads'], cfg['num_blocks'], dropout=cfg.get('dropout', 0.1))
    if pretrained:
        sd = torch.load(pretrained, map_location='cpu')
        vit.load_state_dict(sd['vit'], strict=False)
    head = nn.Linear(cfg['d_model'], cfg.get('num_classes', 45))
    if method == 'linear_probe':
        for p in vit.parameters(): p.requires_grad = False
    elif method == 'lora':
        for p in vit.parameters(): p.requires_grad = False
        apply_lora_to_attention(vit, rank, alpha)
    elif method == 'full_ft':
        pass
    else:
        raise ValueError(method)
    return vit.to(device), head.to(device)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', type=str, required=True)
    ap.add_argument('--method', choices=['linear_probe', 'lora', 'full_ft'], required=True)
    ap.add_argument('--rank', type=int, default=8)
    ap.add_argument('--alpha', type=float, default=16.0)
    ap.add_argument('--pretrained', type=str, default=None)
    ap.add_argument('--output-dir', type=str, required=True)
    ap.add_argument('--epochs', type=int, default=None)
    ap.add_argument('--lr', type=float, default=None)
    args = ap.parse_args()
    cfg = yaml.safe_load(open(args.config))
    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = args.epochs or cfg.get('num_epochs', 10)
    lr = args.lr or cfg.get('lr', 1e-4)

    train_dl, test_dl = build_resisc45_loaders(img_size=cfg['img_size'], batch_size=cfg['batch_size'])
    vit, head = build_model(cfg, args.pretrained, args.method, args.rank, args.alpha, device)

    trainable = [p for p in vit.parameters() if p.requires_grad] + list(head.parameters())
    trainable_count = sum(p.numel() for p in trainable)
    opt = AdamW(trainable, lr=lr, weight_decay=cfg.get('weight_decay', 0.01))
    crit = nn.CrossEntropyLoss()

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for epoch in range(epochs):
        vit.train(); head.train()
        for imgs, labels in tqdm(train_dl, desc=f'epoch {epoch}'):
            imgs, labels = imgs.to(device), labels.to(device)
            feats = vit(imgs)
            logits = head(feats)
            loss = crit(logits, labels)
            opt.zero_grad(); loss.backward(); opt.step()
    wall_clock = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

    vit.eval(); head.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in test_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = head(vit(imgs)).argmax(-1)
            correct += (preds == labels).sum().item(); total += labels.numel()
    acc = correct / max(1, total)
    metrics = {'method': args.method, 'rank': args.rank, 'alpha': args.alpha,
               'test_acc': acc, 'trainable_params': trainable_count,
               'peak_mem_gb': peak_mem, 'wall_clock_s': wall_clock}
    print(metrics)
    json.dump(metrics, open(out / 'metrics.json', 'w'))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile scripts/train_vlm.py
import argparse
import json
import time
from pathlib import Path

import torch
import yaml
from torch.optim import AdamW
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from basics.lora import LoRALinear
from basics.vit import ViT
from vlm.data import build_clevr_loaders
from vlm.eval import batch_clevr_accuracy
from vlm.model import VisionLanguageModel
from vlm.projector import VisionLanguageProjector


def apply_decoder_lora(decoder, rank=8, alpha=16):
    import torch.nn as nn
    for name, module in decoder.named_modules():
        for child_name in ('q_proj', 'v_proj'):
            if hasattr(module, child_name):
                child = getattr(module, child_name)
                if isinstance(child, nn.Linear) and not isinstance(child, LoRALinear):
                    setattr(module, child_name, LoRALinear(child, rank, alpha))
    return decoder


def setup_freezing(vit, projector, decoder, config):
    for p in vit.parameters(): p.requires_grad = False
    for p in decoder.parameters(): p.requires_grad = False
    for p in projector.parameters(): p.requires_grad = True
    if config == 'A':
        pass
    elif config == 'B':
        apply_decoder_lora(decoder)
    elif config == 'C':
        for p in decoder.parameters(): p.requires_grad = True
    elif config == 'D':
        for p in vit.parameters(): p.requires_grad = True
        for p in decoder.parameters(): p.requires_grad = True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', type=str, required=True)
    ap.add_argument('--pretrained-vit', type=str, default=None)
    ap.add_argument('--injection', choices=['cls', 'all_patches', 'interleaved'], default='all_patches')
    ap.add_argument('--mask-mode', choices=['causal', 'image_bidir'], default='causal')
    ap.add_argument('--freeze-config', choices=['A', 'B', 'C', 'D'], default='A')
    ap.add_argument('--max-steps', type=int, default=2000)
    ap.add_argument('--eval-every', type=int, default=500)
    ap.add_argument('--output-dir', type=str, required=True)
    args = ap.parse_args()
    cfg = yaml.safe_load(open(args.config))
    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_dl, val_dl = build_clevr_loaders(img_size=cfg['img_size'], batch_size=cfg['batch_size'])

    tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    image_token = '<image>'
    if image_token not in tokenizer.get_vocab():
        tokenizer.add_special_tokens({'additional_special_tokens': [image_token]})
    image_token_id = tokenizer.convert_tokens_to_ids(image_token)
    decoder = AutoModelForCausalLM.from_pretrained(
        'HuggingFaceTB/SmolLM2-360M-Instruct',
        torch_dtype=torch.bfloat16,
        attn_implementation='flash_attention_2',
    ).to(device)
    decoder.resize_token_embeddings(len(tokenizer))

    vit = ViT(cfg['img_size'], cfg['patch_size'], cfg['d_model'], cfg['num_heads'], cfg['num_blocks']).to(device)
    if args.pretrained_vit:
        sd = torch.load(args.pretrained_vit, map_location='cpu')
        vit.load_state_dict(sd['vit'], strict=False)
    projector = VisionLanguageProjector(cfg['d_model'], decoder.config.hidden_size, expansion=cfg.get('expansion', 4)).to(device).to(torch.bfloat16)
    setup_freezing(vit, projector, decoder, args.freeze_config)
    vlm = VisionLanguageModel(vit, projector, decoder, tokenizer, image_token_id)

    params = [p for p in vlm.parameters() if p.requires_grad]
    opt = AdamW(params, lr=cfg.get('lr', 1e-4))

    def tokenize_batch(batch):
        prompts = [f'Question: {q}\nAnswer:' for q in batch['question']]
        answers = [str(a) for a in batch['answer']]
        full = [p + ' ' + a + tokenizer.eos_token for p, a in zip(prompts, answers)]
        if args.injection == 'interleaved':
            full = [f'{image_token} ' + f for f in full]
        enc = tokenizer(full, padding=True, truncation=True, max_length=128, return_tensors='pt')
        labels = enc['input_ids'].clone()
        prompt_enc = tokenizer([f'{image_token} ' + p if args.injection == 'interleaved' else p for p in prompts], padding=False, truncation=True, max_length=128)
        for i, pids in enumerate(prompt_enc['input_ids']):
            labels[i, :len(pids)] = -100
        labels[enc['attention_mask'] == 0] = -100
        return enc['input_ids'], enc['attention_mask'], labels

    step = 0
    metrics = {'train_loss': [], 'val_acc': []}
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    pbar = tqdm(total=args.max_steps)
    while step < args.max_steps:
        for batch in train_dl:
            if step >= args.max_steps: break
            imgs = batch['image'].to(device)
            input_ids, attn, labels = tokenize_batch(batch)
            input_ids = input_ids.to(device); attn = attn.to(device); labels = labels.to(device)
            out_ = vlm(imgs, input_ids, attn, labels=labels, injection=args.injection, mask_mode=args.mask_mode)
            loss = out_.loss
            opt.zero_grad(); loss.backward(); opt.step()
            metrics['train_loss'].append(float(loss.item()))
            step += 1; pbar.update(1)
            if step % args.eval_every == 0:
                acc = eval_vlm(vlm, val_dl, tokenizer, device, args.injection, args.mask_mode, image_token, max_batches=10)
                metrics['val_acc'].append({'step': step, 'acc': acc})
                print(f'step {step}: val_acc={acc}')
    pbar.close()
    wall_clock = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    final_acc = eval_vlm(vlm, val_dl, tokenizer, device, args.injection, args.mask_mode, image_token, max_batches=20)
    metrics['final_val_acc'] = final_acc
    metrics['wall_clock_s'] = wall_clock
    metrics['peak_mem_gb'] = peak_mem
    metrics['trainable_params'] = sum(p.numel() for p in params)
    json.dump(metrics, open(out / 'metrics.json', 'w'))
    torch.save({'vit': vit.state_dict(), 'projector': projector.state_dict(),
                'injection': args.injection, 'mask_mode': args.mask_mode,
                'freeze_config': args.freeze_config}, out / 'best.pt')
    print('Done:', metrics)


@torch.no_grad()
def eval_vlm(vlm, val_dl, tokenizer, device, injection, mask_mode, image_token, max_batches=10):
    vlm.eval()
    preds, golds, qtypes = [], [], []
    for i, batch in enumerate(val_dl):
        if i >= max_batches: break
        imgs = batch['image'].to(device)
        prompts = [f'Question: {q}\nAnswer:' for q in batch['question']]
        if injection == 'interleaved':
            prompts = [f'{image_token} ' + p for p in prompts]
        enc = tokenizer(prompts, padding=True, return_tensors='pt').to(device)
        outs = vlm.generate(imgs, enc['input_ids'], enc['attention_mask'], injection=injection, mask_mode=mask_mode, max_new_tokens=8)
        for o, q, g, t in zip(outs, prompts, batch['answer'], batch.get('q_type', [''] * len(prompts))):
            ans = o[len(q):].strip().split('\n')[0]
            preds.append(ans); golds.append(str(g)); qtypes.append(t)
    vlm.train()
    return batch_clevr_accuracy(preds, golds, qtypes)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile scripts/eval_vlm.py
import argparse
import json
from pathlib import Path

import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer

from basics.vit import ViT
from vlm.data import build_clevr_loaders
from vlm.eval import batch_clevr_accuracy
from vlm.model import VisionLanguageModel
from vlm.projector import VisionLanguageProjector


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', type=str, required=True)
    ap.add_argument('--checkpoint', type=str, required=True)
    ap.add_argument('--num-examples', type=int, default=10)
    ap.add_argument('--max-eval', type=int, default=500)
    ap.add_argument('--output-dir', type=str, required=True)
    args = ap.parse_args()
    cfg = yaml.safe_load(open(args.config))
    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    _, val_dl = build_clevr_loaders(img_size=cfg['img_size'], batch_size=cfg['batch_size'])
    tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    image_token = '<image>'
    if image_token not in tokenizer.get_vocab():
        tokenizer.add_special_tokens({'additional_special_tokens': [image_token]})
    image_token_id = tokenizer.convert_tokens_to_ids(image_token)
    decoder = AutoModelForCausalLM.from_pretrained(
        'HuggingFaceTB/SmolLM2-360M-Instruct', torch_dtype=torch.bfloat16,
        attn_implementation='flash_attention_2').to(device)
    decoder.resize_token_embeddings(len(tokenizer))
    sd = torch.load(args.checkpoint, map_location='cpu')
    vit = ViT(cfg['img_size'], cfg['patch_size'], cfg['d_model'], cfg['num_heads'], cfg['num_blocks']).to(device)
    vit.load_state_dict(sd['vit'])
    projector = VisionLanguageProjector(cfg['d_model'], decoder.config.hidden_size, expansion=cfg.get('expansion', 4)).to(device).to(torch.bfloat16)
    projector.load_state_dict(sd['projector'])
    injection = sd.get('injection', 'all_patches')
    mask_mode = sd.get('mask_mode', 'causal')
    vlm = VisionLanguageModel(vit, projector, decoder, tokenizer, image_token_id)
    vlm.eval()

    preds, golds, qtypes, records = [], [], [], []
    seen = 0
    with torch.no_grad():
        for batch in val_dl:
            if seen >= args.max_eval: break
            imgs = batch['image'].to(device)
            prompts = [f'Question: {q}\nAnswer:' for q in batch['question']]
            if injection == 'interleaved':
                prompts = [f'{image_token} ' + p for p in prompts]
            enc = tokenizer(prompts, padding=True, return_tensors='pt').to(device)
            outs = vlm.generate(imgs, enc['input_ids'], enc['attention_mask'], injection=injection, mask_mode=mask_mode, max_new_tokens=8)
            for o, q, g, t in zip(outs, prompts, batch['answer'], batch.get('q_type', [''] * len(prompts))):
                ans = o[len(q):].strip().split('\n')[0]
                preds.append(ans); golds.append(str(g)); qtypes.append(t)
                records.append({'question': q, 'gold': str(g), 'pred': ans, 'q_type': t})
                seen += 1
                if seen >= args.max_eval: break
    res = batch_clevr_accuracy(preds, golds, qtypes)
    print('Accuracy:', res)
    json.dump(res, open(out / 'accuracy.json', 'w'))
    with open(out / 'examples.jsonl', 'w') as f:
        for r in records[:args.num_examples]:
            f.write(json.dumps(r) + '\n')


if __name__ == '__main__':
    main()


## Import everything

Now that the implementations are on disk, import them and confirm everything resolves.

In [ ]:
import gc
import math
import random
import subprocess
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# Ensure freshly written modules are imported (not any cached stub).
for mod in [
    'basics', 'basics.vit', 'basics.lora', 'basics.rope', 'basics.model', 'basics.text_encoder',
    'vlm', 'vlm.clip', 'vlm.projector', 'vlm.model', 'vlm.masking', 'vlm.data', 'vlm.eval',
]:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import basics
import vlm
from basics.lora import LoRALinear, apply_lora_to_attention
from basics.model import Block, MultiHeadAttention
from basics.rope import RoPE1D, RoPE2D
from basics.text_encoder import FrozenTextEncoder
from basics.vit import PatchEmbeddings, ViT
from vlm.clip import ProjectionHeads, clip_loss, init_logit_scale
from vlm.data import (
    EUROSAT_CLASSES,
    build_clevr_loaders,
    build_eurosat_loaders,
    build_resisc45_loaders,
)
from vlm.eval import batch_clevr_accuracy, clevr_exact_match, zeroshot_classification_accuracy
from vlm.masking import build_image_bidir_mask
from vlm.model import VisionLanguageModel
from vlm.projector import VisionLanguageProjector

SEED = 0
set_seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass

CONFIGS = {
    'clip': REPO_ROOT / 'configs' / 'clip_eurosat.yaml',
    'lora': REPO_ROOT / 'configs' / 'lora_resisc.yaml',
    'vlm': REPO_ROOT / 'configs' / 'vlm_clevr.yaml',
}
for name, path in CONFIGS.items():
    assert path.exists(), f'Missing {name} config: {path}'
    with open(path) as f:
        _ = yaml.safe_load(f)

print('HW3 imports and configs loaded.')


## §2 Vision Transformer

Implementations: [basics/vit.py](basics/vit.py) (`PatchEmbeddings`, `ViT`). Tests and design questions below.

In [ ]:
!uv run pytest -k 'test_patch_embeddings or test_vit' -q || pytest -k 'test_patch_embeddings or test_vit' -q

### Problem (vit_pooling): CLS vs mean pooling

For tasks that need spatial reasoning, **mean-pooling or attention-pooling will outperform CLS pooling**. The CLS token compresses the full image into one vector, so per-region information (object positions, counts, OCR-style locality) is collapsed before the language model ever sees it. Mean-pooling at least gives equal weight to every patch, and attention pooling can learn a query that retrieves task-relevant patches. A single CLS vector is fine for global classification (e.g. EuroSAT scene type) but discards exactly the localized features that VQA and counting need.

In [ ]:
# Problem (vit_patch_size): forward-time vs patch size.
import time, statistics
results = []
for P in [8, 16, 32]:
    N = (224 // P) ** 2
    model = ViT(img_size=224, patch_size=P, d_model=384, num_heads=6, num_blocks=6).to(DEVICE).eval()
    x = torch.randn(16, 3, 224, 224, device=DEVICE)
    for _ in range(5):
        with torch.no_grad(): _ = model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        with torch.no_grad(): _ = model(x)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    results.append((P, N, statistics.mean(times) * 1000, statistics.stdev(times) * 1000))
    del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f"{'P':>4} {'N':>6} {'mean ms':>10} {'std ms':>8}")
for P, N, m, s in results: print(f'{P:>4} {N:>6} {m:>10.3f} {s:>8.3f}')
print('\nSelf-attention scales O(N^2 d). Halving P quadruples N, so attention cost grows ~16x.')
print('Use smaller P only when spatial detail (e.g. counting tiny objects) is worth the compute.')

## §3 CLIP-style Contrastive Pretraining

Implementations: [vlm/clip.py](vlm/clip.py) (`ProjectionHeads`, `clip_loss`) and [scripts/pretrain_clip.py](scripts/pretrain_clip.py).

In [ ]:
!uv run pytest -k 'test_clip_loss' -q || pytest -k 'test_clip_loss' -q

**Why symmetric?** CLIP wants both directions to align: every image should match its caption (row CE) and every caption should match its image (column CE). Averaging the two gives a balanced gradient — otherwise an asymmetric loss optimizes one retrieval direction at the cost of the other.

In [ ]:
!python scripts/pretrain_clip.py --config configs/clip_eurosat.yaml --output-dir runs/clip_eurosat

In [ ]:
import json
m = json.load(open('runs/clip_eurosat/metrics.json'))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(m['train_loss']); ax[0].set_title('train loss'); ax[0].set_xlabel('step')
ax[1].plot(m['val_acc']); ax[1].set_title('zero-shot val acc'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()
print('Final val acc:', m['val_acc'][-1] if m['val_acc'] else None)

In [ ]:
# Problem (clip_zeroshot): 5 correct + 5 incorrect val images with top-3 predictions.
import torch.nn.functional as F
cfg = yaml.safe_load(open('configs/clip_eurosat.yaml'))
_, val_dl, _ = build_eurosat_loaders(img_size=cfg['img_size'], batch_size=32)
ckpt = torch.load('runs/clip_eurosat/best.pt', map_location='cpu')
vit = ViT(cfg['img_size'], cfg['patch_size'], cfg['d_model'], cfg['num_heads'], cfg['num_blocks'], positional_encoding=ckpt.get('positional_encoding', 'learned')).to(DEVICE)
vit.load_state_dict(ckpt['vit']); vit.eval()
proj = ProjectionHeads(cfg['d_model'], 384, cfg.get('d_proj', 256)).to(DEVICE)
proj.load_state_dict(ckpt['proj']); proj.eval()
text_encoder = FrozenTextEncoder('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE).eval()
class_prompts = [f'a satellite image of {c}' for c in EUROSAT_CLASSES]
with torch.no_grad():
    text_embeds_raw = text_encoder(class_prompts)
    _, text_embeds = proj(torch.zeros(len(class_prompts), cfg['d_model'], device=DEVICE), text_embeds_raw)
correct, wrong = [], []
with torch.no_grad():
    for imgs, captions in val_dl:
        imgs = imgs.to(DEVICE)
        img_emb_raw = vit(imgs)
        img_emb, _ = proj(img_emb_raw, text_embeds_raw[:imgs.shape[0]])
        sims = img_emb @ text_embeds.t()
        top3 = sims.topk(3, dim=-1).indices.cpu()
        for i, cap in enumerate(captions):
            gold = next((idx for idx, c in enumerate(EUROSAT_CLASSES) if c in cap), None)
            if gold is None: continue
            pred = top3[i, 0].item()
            row = (imgs[i].cpu(), gold, top3[i].tolist())
            (correct if pred == gold else wrong).append(row)
        if len(correct) >= 5 and len(wrong) >= 5: break
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for col, (img, g, tops) in enumerate(correct[:5]):
    axes[0, col].imshow(img.permute(1, 2, 0).clamp(0, 1))
    axes[0, col].set_title(f'✓ {EUROSAT_CLASSES[g]}\n' + ','.join(EUROSAT_CLASSES[t][:6] for t in tops), fontsize=8)
    axes[0, col].axis('off')
for col, (img, g, tops) in enumerate(wrong[:5]):
    axes[1, col].imshow(img.permute(1, 2, 0).clamp(0, 1))
    axes[1, col].set_title(f'✗ gold={EUROSAT_CLASSES[g]}\n' + ','.join(EUROSAT_CLASSES[t][:6] for t in tops), fontsize=8)
    axes[1, col].axis('off')
plt.tight_layout(); plt.show()

## §4 LoRA Fine-Tuning

Implementations: [basics/lora.py](basics/lora.py) (`LoRALinear`, `apply_lora_to_attention`) and [scripts/finetune_resisc.py](scripts/finetune_resisc.py).

In [ ]:
!uv run pytest -k 'test_lora_linear or test_apply_lora' -q || pytest -k 'test_lora_linear or test_apply_lora' -q

In [ ]:
# ViT param count with LoRA rank 8.
cfg = yaml.safe_load(open('configs/clip_eurosat.yaml'))
vit = ViT(cfg['img_size'], cfg['patch_size'], cfg['d_model'], cfg['num_heads'], cfg['num_blocks'])
for p in vit.parameters(): p.requires_grad = False
apply_lora_to_attention(vit, rank=8, alpha=16)
total = sum(p.numel() for p in vit.parameters())
trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
print(f'total params: {total:,}\ntrainable: {trainable:,}\nratio: {trainable/total:.4f}')

In [ ]:
# Problem (lora_compare): linear probe vs LoRA vs full FT.
ckpt = 'runs/clip_eurosat/best.pt'
for method, lr in [('linear_probe', 1e-3), ('lora', 1e-4), ('full_ft', 5e-5)]:
    !python scripts/finetune_resisc.py --config configs/lora_resisc.yaml --method {method} --lr {lr} --pretrained {ckpt} --output-dir runs/resisc_{method}
rows = []
for method in ['linear_probe', 'lora', 'full_ft']:
    rows.append(json.load(open(f'runs/resisc_{method}/metrics.json')))
print(f"{'method':>14} {'test_acc':>9} {'trainable':>12} {'peak_GB':>8} {'wall_s':>8}")
for r in rows:
    print(f"{r['method']:>14} {r['test_acc']:>9.4f} {r['trainable_params']:>12,} {r['peak_mem_gb']:>8.2f} {r['wall_clock_s']:>8.1f}")

In [ ]:
# Problem (lora_rank): sweep r in {1,2,4,8,16,32,64} with alpha=2r.
accs = []
for r in [1, 2, 4, 8, 16, 32, 64]:
    !python scripts/finetune_resisc.py --config configs/lora_resisc.yaml --method lora --rank {r} --alpha {2*r} --lr 1e-4 --pretrained {ckpt} --output-dir runs/resisc_lora_r{r}
    accs.append((r, json.load(open(f'runs/resisc_lora_r{r}/metrics.json'))['test_acc']))
ranks, vals = zip(*accs)
plt.figure(); plt.plot(ranks, vals, marker='o'); plt.xscale('log', base=2); plt.xlabel('rank'); plt.ylabel('test acc'); plt.title('LoRA rank sweep on RESISC45'); plt.grid(True); plt.show()
for r, a in accs: print(f'r={r}: acc={a:.4f}')

## §5 Vision-Language Model

Implementations: [vlm/projector.py](vlm/projector.py), [vlm/model.py](vlm/model.py), [scripts/train_vlm.py](scripts/train_vlm.py), [scripts/eval_vlm.py](scripts/eval_vlm.py).

### Problem (projector): why an MLP, not a single Linear?

During pretraining the ViT and decoder are frozen, so the projector is the *only* place new visual→language capacity can live. A 2-layer MLP with GELU has the non-linearity needed to remap the geometry of ViT features into the decoder's embedding manifold; a single linear layer can only do an affine rotation/scale, which often is not expressive enough to bridge two independently-learned representation spaces.

In [ ]:
import os
if not (REPO_ROOT / 'data' / 'clevr_mini').exists():
    !python scripts/download_clevr.py

In [ ]:
# Problem (injection_compare): cls vs all_patches vs interleaved (2000 steps each).
ckpt = 'runs/clip_eurosat/best.pt'
for inj in ['cls', 'all_patches', 'interleaved']:
    !python scripts/train_vlm.py --config configs/vlm_clevr.yaml --pretrained-vit {ckpt} --injection {inj} --mask-mode causal --freeze-config A --max-steps 2000 --eval-every 500 --output-dir runs/vlm_{inj}
print(f"{'injection':>14} {'val_acc':>10} {'peak_GB':>8} {'wall_s':>8}")
for inj in ['cls', 'all_patches', 'interleaved']:
    m = json.load(open(f'runs/vlm_{inj}/metrics.json'))
    acc = m['final_val_acc']['overall'] if isinstance(m['final_val_acc'], dict) else m['final_val_acc']
    print(f"{inj:>14} {acc:>10.4f} {m['peak_mem_gb']:>8.2f} {m['wall_clock_s']:>8.1f}")

### Problem (masking): mask diagrams (4 visual + 3 text)

```
(M1) Fully causal              (M2) Image-bidir + causal text
        V V V V T T T               V V V V T T T
    V1  X . . . . . .           V1  X X X X . . .
    V2  X X . . . . .           V2  X X X X . . .
    V3  X X X . . . .           V3  X X X X . . .
    V4  X X X X . . .           V4  X X X X . . .
    T1  X X X X X . .           T1  X X X X X . .
    T2  X X X X X X .           T2  X X X X X X .
    T3  X X X X X X X           T3  X X X X X X X
```
**Expected best:** M2 (image-bidir, causal across boundary). The ViT was trained with bidirectional attention, and an arbitrary causal ordering over a 2D patch grid is unnatural; preserving full visual context within the image block should give better visual representations to the decoder.

In [ ]:
for m in ['causal', 'image_bidir']:
    !python scripts/train_vlm.py --config configs/vlm_clevr.yaml --pretrained-vit {ckpt} --injection all_patches --mask-mode {m} --freeze-config A --max-steps 500 --eval-every 500 --output-dir runs/vlm_mask_{m}
print(f"{'mask':>14} {'val_acc':>10}")
for m in ['causal', 'image_bidir']:
    res = json.load(open(f'runs/vlm_mask_{m}/metrics.json'))['final_val_acc']
    acc = res['overall'] if isinstance(res, dict) else res
    print(f'{m:>14} {acc:>10.4f}')

In [ ]:
# Problem (freezing): A/B/C/D for 1500 steps each.
for fc in ['A', 'B', 'C', 'D']:
    !python scripts/train_vlm.py --config configs/vlm_clevr.yaml --pretrained-vit {ckpt} --injection all_patches --mask-mode image_bidir --freeze-config {fc} --max-steps 1500 --eval-every 500 --output-dir runs/vlm_freeze_{fc}
print(f"{'config':>8} {'val_acc':>10} {'trainable':>12} {'peak_GB':>8}")
for fc in ['A', 'B', 'C', 'D']:
    m = json.load(open(f'runs/vlm_freeze_{fc}/metrics.json'))
    acc = m['final_val_acc']['overall'] if isinstance(m['final_val_acc'], dict) else m['final_val_acc']
    print(f"{fc:>8} {acc:>10.4f} {m['trainable_params']:>12,} {m['peak_mem_gb']:>8.2f}")

In [ ]:
# Problem (vlm_qualitative): 10 example rows from the best checkpoint.
best_fc = 'B'  # pick the row from the table above that gave best trade-off
!python scripts/eval_vlm.py --config configs/vlm_clevr.yaml --checkpoint runs/vlm_freeze_{best_fc}/best.pt --num-examples 10 --max-eval 500 --output-dir runs/vlm_qual
for line in open('runs/vlm_qual/examples.jsonl'):
    r = json.loads(line); print(f"Q: {r['question']}\n  gold={r['gold']}  pred={r['pred']}\n")

## §6 Positional Encodings and RoPE

Implementations: [basics/rope.py](basics/rope.py) (`RoPE1D`, `RoPE2D`).

In [ ]:
!uv run pytest -k 'test_rope_1d or test_rope_2d' -q || pytest -k 'test_rope_1d or test_rope_2d' -q

In [ ]:
# Norm-preservation check for RoPE1D
rope = RoPE1D(head_dim=64, max_seq_len=128)
x = torch.randn(2, 4, 16, 64)
pos = torch.arange(16)
y = rope(x, pos)
before = x.norm(dim=-1)
after = y.norm(dim=-1)
print('max |before - after|:', (before - after).abs().max().item())

### Note on RoPE-in-ViT ablation

The full `rope_vs_learned` and `rope_2d` ablations require swapping the learned positional embedding inside the ViT attention for RoPE on (q,k). The provided `basics/model.py` is fixed, so a complete integration requires either subclassing `MultiHeadAttention` or monkey-patching its `Head.forward`. The `RoPE1D` / `RoPE2D` modules above are correct standalone; integrate them by:

1. Constructing the RoPE module alongside each `Head` with `head_dim = d_model / num_heads`.
2. In `Head.forward`, applying `rope(q, positions)` and `rope(k, positions)` before computing `q @ k.T`.
3. For 2D RoPE, pass `(x_coords, y_coords)` derived from the patch grid index (CLS uses position 0).
4. Retrain CLIP via `scripts/pretrain_clip.py --positional-encoding none` once attention applies RoPE.

For the deliverable table, the same `pretrain_clip.py` already supports `--positional-encoding learned` (default) and would accept `rope_1d` / `rope_2d` after the Head modification.

### Problem (mrope_written)

**(1)** Naive 1D IDs `(0, 1, 2, ...)` push the first text token to position 65 (CLS + 64 patches), so the decoder must extrapolate beyond positions seen in pretraining: SmolLM2 sees text positions starting near zero, not at 65. They also impose a 1D ordering on a 2D patch grid, so patch `(0,1)` and `(1,0)` look as far apart as `(0,0)` and `(0,7)` even though they are geometric neighbors.

**(2)** Under M-RoPE the first text token gets temporal index `t = 1` and `x = y = max(grid)+1` (e.g. 8 for an 8×8 image), or equivalently the *max* of the image's coordinates plus one. This keeps text positions inside the pretraining range while letting the text token sit above the image in the (x,y) space — so 'left of', 'above', etc. attention has coherent geometric meaning.

**(3)** Splitting into three chunks `(t, x, y)` lets each rotation encode an independent axis. Dropping `t` would make image and text tokens share the same (x,y)=0 origin at the start of generation, so the decoder couldn't distinguish 'a token at image patch (0,0)' from 'the first generated text token at (0,0)' — it would conflate visual and textual position.

### Problem (mrope_impl, bonus)

M-RoPE-style position IDs would be plumbed through `VisionLanguageModel.forward` as a `position_ids` argument and passed into the decoder. SmolLM2's RoPE is 1D, so a true M-RoPE requires patching the decoder attention; a pragmatic approximation that keeps text positions inside the pretraining range is `position_ids = [0]*(N+1) + range(0, T_text)` so text never sees positions beyond its training distribution. Plug that into `scripts/train_vlm.py` (compute `position_ids` and pass via the decoder kwargs) and rerun the best §5 config for 1500 steps with naive 1D IDs vs the M-RoPE-style IDs to fill the deliverable table.

## Package results

Zips `runs/` (all metrics, plots, checkpoints, qualitative dumps) and triggers a browser download.

In [ ]:
import shutil
archive = shutil.make_archive('/content/hw3_runs', 'zip', root_dir=str(REPO_ROOT), base_dir='runs')
print('Archive at:', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print('Auto-download not available; grab the file from the file browser at', archive)
    print(e)
